# Financial Transactions Dataset: Analytics

Source :
https://www.kaggle.com/datasets/computingvictor/transactions-fraud-datasets/data




## Load Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q pyspark orjson

In [ ]:
import json
import os
from pathlib import Path

import joblib
import numpy as np
import orjson
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    sum,
    count,
    expr,
    regexp_replace,
    round,
    sum as spark_sum,
    to_timestamp,
    when,
)
from pyspark.sql.types import DoubleType, IntegerType

from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    ConfusionMatrixDisplay,
    f1_score,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.preprocessing import (
    LabelEncoder,
    OrdinalEncoder,
    StandardScaler,
)

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    pipeline,
)

from xgboost import XGBClassifier

In [ ]:
spark = (
    SparkSession.builder
    .appName("PermataBank_Fraud")
    .getOrCreate()
)

spark.catalog.clearCache()

In [ ]:
users_data = (
    spark.read
    .option("header", True)
    .csv("/content/drive/MyDrive/archive/users_data.csv")
)

cards_data = (
    spark.read
    .option("header", True)
    .csv("/content/drive/MyDrive/archive/cards_data.csv")
)

transactions_data = (
    spark.read
    .option("header", True)
    .csv("/content/drive/MyDrive/archive/transactions_data.csv")
)

print("Users")
users_data.printSchema()

print("Cards")
cards_data.printSchema()

print("Transactions")
transactions_data.printSchema()

users_data.show(5, truncate=False)
cards_data.show(5, truncate=False)
transactions_data.show(5, truncate=False)


In [ ]:
with open("/content/drive/MyDrive/archive/mcc_codes.json") as f:
    mcc = json.load(f)

df_mcc = spark.createDataFrame(
    list(mcc.items()),
    ["mcc_code", "mcc_description"]
)

print("MCC")
df_mcc.printSchema()
df_mcc.show(5, truncate=False)


with open("/content/drive/MyDrive/archive/train_fraud_labels.json", "rb") as f:
    labels = orjson.loads(f.read())

df_target = spark.createDataFrame(
    list(labels["target"].items()),
    ["transaction_id", "label"]
)

print("Fraud Labels")
df_target.printSchema()
df_target.show(5, truncate=False)


print("Users:", users_data.count())
print("Cards:", cards_data.count())
print("Transactions:", transactions_data.count())
print("MCC:", df_mcc.count())
print("Labels:", df_target.count())

## Data Pre-processing

### Missing Value

Based on this process, the missing value are merchant_state,zip, errors

In [ ]:

users_null_counts = users_data.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in users_data.columns
])

users_null_counts.show(truncate=False)

cards_null_counts = cards_data.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in cards_data.columns
])

cards_null_counts.show(truncate=False)

transactions_null_counts = transactions_data.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in transactions_data.columns
])

transactions_null_counts.show(truncate=False)

#### Handle Errors null

> Because transaction is successful, the error field is null, so replace it with "No Error".

In [ ]:
transactions_data.groupBy("errors") \
    .count() \
    .orderBy(col("count").desc()) \
    .show(truncate=False)

In [ ]:
transactions_data = transactions_data.fillna({"errors": "No Error"})

#### Handle merchant_state, zip

> Because these are online transactions, the merchant state, city, and ZIP code are unavailable. Missing values are filled with:
>
> * `"merchant_state": "OL"`
> * `"merchant_city": "Online"`
> * `"zip": "0"`


In [ ]:
missing_state = transactions_data.filter(
    col("merchant_state").isNull() | (col("merchant_state") == "")
)

print("Total missing merchant_state:", missing_state.count())

online_missing = missing_state.filter(
    col("merchant_city") == "ONLINE"
)

print("Missing merchant_state with merchant_city = ONLINE:", online_missing.count())

offline_missing = missing_state.filter(
    col("merchant_city") != "ONLINE"
)

print("Missing merchant_state with merchant_city != ONLINE:", offline_missing.count())

offline_missing_zip_null = missing_state.filter(
    (col("merchant_city") != "ONLINE") &
    (col("zip").isNull())
)

print("Missing merchant_state, non-ONLINE, ZIP is NULL:", offline_missing_zip_null.count())

print(
    "merchant_state is NULL:",
    transactions_data.filter(col("merchant_state").isNull()).count()
)

In [ ]:
total_missing = missing_state.count()
online_missing = missing_state.filter(
    col("merchant_city") == "ONLINE"
).count()

print(f"Total missing merchant_state: {total_missing}")
print(f"ONLINE missing merchant_state: {online_missing}")

if total_missing == online_missing:
    print("All missing merchant_state values are from ONLINE transactions.")
else:
    print(f" {total_missing - online_missing} missing merchant_state records are NOT ONLINE transactions.")

In [ ]:
from pyspark.sql.functions import sum, count

transactions_data.groupBy("merchant_state","merchant_city") \
    .agg(
        count("*").alias("transaction_count")
            ) \
    .orderBy(col("transaction_count").desc()) \
    .show(truncate=False)

In [ ]:
transactions_data = transactions_data.fillna({
    "merchant_state": "OL",
    "merchant_city": "Online",
    "zip": "0"
})

### Data Cleaning

> Remove $ sign , and cast data to int type

In [ ]:


users_data = (
    users_data
    .withColumn("id", col("id").cast(IntegerType()))
    .withColumn("current_age", col("current_age").cast(IntegerType()))
    .withColumn("retirement_age", col("retirement_age").cast(IntegerType()))
    .withColumn("birth_year", col("birth_year").cast(IntegerType()))
    .withColumn("birth_month", col("birth_month").cast(IntegerType()))
    .withColumn("credit_score", col("credit_score").cast(IntegerType()))
    .withColumn("num_credit_cards", col("num_credit_cards").cast(IntegerType()))

    .withColumn(
        "per_capita_income",
        regexp_replace(
            regexp_replace(col("per_capita_income"), "\\$", ""),
            ",", ""
        ).cast(DoubleType())
    )
    .withColumn(
        "yearly_income",
        regexp_replace(
            regexp_replace(col("yearly_income"), "\\$", ""),
            ",", ""
        ).cast(DoubleType())
    )
    .withColumn(
        "total_debt",
        regexp_replace(
            regexp_replace(col("total_debt"), "\\$", ""),
            ",", ""
        ).cast(DoubleType())
    )
)

users_data.printSchema()
users_data.show(5, truncate=False)

In [ ]:
cards_data = (
    cards_data
    .withColumn("id", col("id").cast(IntegerType()))
    .withColumn("client_id", col("client_id").cast(IntegerType()))
    .withColumn("year_pin_last_changed", col("year_pin_last_changed").cast(IntegerType()))
    .withColumn(
        "credit_limit",
        regexp_replace(
            regexp_replace(col("credit_limit"), "\\$", ""),
            ",", ""
        ).cast(DoubleType())
    )
)

cards_data.printSchema()
cards_data.show(5, truncate=False)

In [ ]:

transactions_data = (
    transactions_data
    .withColumn("id", col("id").cast(IntegerType()))
    .withColumn("client_id", col("client_id").cast(IntegerType()))
    .withColumn("card_id", col("card_id").cast(IntegerType()))
    .withColumn("merchant_id", col("merchant_id").cast(IntegerType()))
    .withColumn("mcc", col("mcc").cast(IntegerType()))
    .withColumn(
        "date",
        to_timestamp(col("date"), "yyyy-MM-dd HH:mm:ss")
    )

    .withColumn(
        "amount",
        regexp_replace(
            regexp_replace(col("amount"), "\\$", ""),
            ",", ""
        ).cast(DoubleType())
    )
    .withColumn(
        "zip",
        when(
            col("zip").isNull() | (col("zip") == ""),
            None
        ).otherwise(
            regexp_replace(col("zip"), r"\.0$", "")
        )
    )
    .withColumn(
        "zip",
        col("zip").cast(IntegerType())
    )
)

transactions_data.printSchema()
transactions_data.show(5, truncate=False)

In [ ]:
users_data.write.mode("overwrite").parquet(
    "/content/drive/MyDrive/archive/processed/users"
)

cards_data.write.mode("overwrite").parquet(
    "/content/drive/MyDrive/archive/processed/cards"
)

transactions_data.write.mode("overwrite").parquet(
    "/content/drive/MyDrive/archive/processed/transactions"
)

df_mcc.write.mode("overwrite").parquet(
    "/content/drive/MyDrive/archive/processed/mcc"
)

df_target.write.mode("overwrite").parquet(
    "/content/drive/MyDrive/archive/processed/labels"
)

In [ ]:
transactions_null_counts = transactions_data.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in transactions_data.columns
])

transactions_null_counts.show(truncate=False)

### Data Integration

In [ ]:
users = spark.read.parquet("/content/drive/MyDrive/archive/processed/users")
cards = spark.read.parquet("/content/drive/MyDrive/archive/processed/cards")
transactions = spark.read.parquet("/content/drive/MyDrive/archive/processed/transactions")
mcc = spark.read.parquet("/content/drive/MyDrive/archive/processed/mcc")
labels = spark.read.parquet("/content/drive/MyDrive/archive/processed/labels")

In [ ]:
df = (
    transactions.alias("t")

    # Fraud labels
    .join(
        labels.alias("l"),
        transactions.id == labels.transaction_id,
        "left"
    )

    # Users
    .join(
        users.alias("u"),
        transactions.client_id == users.id,
        "left"
    )

    # Cards
    .join(
        cards.alias("c"),
        (transactions.card_id == cards.id) &
        (transactions.client_id == cards.client_id),
        "left"
    )

    # MCC
    .join(
        mcc.alias("m"),
        transactions.mcc == mcc.mcc_code,
        "left"
    )

    .select(
        "t.*",

        "l.label",

        "u.current_age",
        "u.retirement_age",
        "u.gender",
        "u.per_capita_income",
        "u.yearly_income",
        "u.total_debt",
        "u.credit_score",
        "u.num_credit_cards",

        "c.card_brand",
        "c.card_type",
        "c.has_chip",
        "c.credit_limit",
        "c.card_on_dark_web",

        "m.mcc_description"
    )
)

In [ ]:
df.printSchema()

In [ ]:
df.show(5, truncate=False)

In [ ]:
print("Transactions:", transactions.count())
print("Labels:", labels.count())
print("Merged:", df.count())

In [ ]:
df.groupBy("label").count().show()

####  Null Label
Records with null values in the label column cannot be used for supervised learning. Therefore, they were separated from the labeled dataset.

In [ ]:

labeled_df = df.filter(col("label").isNotNull())

unlabeled_df = df.filter(col("label").isNull())

In [ ]:
labeled_df.write.mode("overwrite").parquet(
    "/content/drive/MyDrive/archive/processed/transaction_label_merge"
)
unlabeled_df.write.mode("overwrite").parquet(
    "/content/drive/MyDrive/archive/processed/transaction_unlabel_merge"
)

In [ ]:
spark.stop()

## Machine Learning

In [ ]:

df = pd.read_parquet("/content/drive/MyDrive/archive/processed/transaction_label_merge")

In [ ]:
df['year'] = df['date'].dt.year

year_label_count = (
    df.groupby(['year', 'label'])
      .size()
      .reset_index(name='count')
)

print(year_label_count)

#### Split Data

> Based on the data, the dataset is split as follows:
>
> * **Training:** 2010–2017
> * **Validation:** 2018
> * **Test:** 2019



In [ ]:
output_dir = Path("/content/drive/MyDrive/archive/data_split")
output_dir.mkdir(parents=True, exist_ok=True)

train_df = df[df["year"].between(2010, 2017)].copy()
valid_df = df[df["year"] == 2018].copy()
test_df = df[df["year"] == 2019].copy()

train_df.to_parquet(output_dir / "train.parquet", index=False)
valid_df.to_parquet(output_dir / "validation.parquet", index=False)
test_df.to_parquet(output_dir / "test.parquet", index=False)

print(f"Train      : {len(train_df):,} rows")
print(f"Validation : {len(valid_df):,} rows")
print(f"Test       : {len(test_df):,} rows")

In [ ]:
output_dir = Path("/content/drive/MyDrive/archive/data_split")

train_df = pd.read_parquet(output_dir / "train.parquet")
valid_df = pd.read_parquet(output_dir / "validation.parquet")
test_df = pd.read_parquet(output_dir / "test.parquet")

In [ ]:
print(train_df["label"].value_counts())
print(valid_df["label"].value_counts())
print(test_df["label"].value_counts())

### EDA

#### Imbalance Dataset

> As shown in the chart, the dataset is highly imbalanced, with far fewer fraudulent transactions than legitimate ones.

In [ ]:

counts = train_df["label"].value_counts().sort_index()
total = counts.sum()

ax = counts.plot(kind="bar")

plt.title("Class Distribution")
plt.xlabel("Label")
plt.ylabel("Number of Transactions")
plt.xticks([0, 1], ["Non-Fraud", "Fraud"], rotation=0)

for i, count in enumerate(counts):
    percentage = count / total * 100
    ax.text(
        i,
        count,
        f"{count:,}\n({percentage:.2f}%)",
        ha="center",
        va="bottom",
        fontsize=10
    )

plt.tight_layout()
plt.show()

#### Fraud Rate

> This chart is used to examine whether fraudulent activities exhibit patterns across different merchant locations.

In [ ]:
train_df["label_num"] = train_df["label"].map({
    "Yes": 1,
    "No": 0
})

location_summary = (
    train_df.groupby("merchant_city")
    .agg(
        Transactions=("label_num", "count"),
        Frauds=("label_num", "sum"),
        Fraud_Rate=("label_num", "mean")
    )
    .reset_index()
)

location_summary = location_summary[
    location_summary["Transactions"] >= 100
]

location_summary["Fraud_Rate (%)"] = (
    location_summary["Fraud_Rate"] * 100
)

top10 = (
    location_summary
    .sort_values("Fraud_Rate (%)", ascending=False)
    .head(10)
    .sort_values("Fraud_Rate (%)")
)

In [ ]:

sns.set_theme(style="whitegrid", font_scale=1.1)

plt.figure(figsize=(10,8))
palette = sns.color_palette("viridis", len(top10))

ax = sns.barplot(
    data=top10.sort_values("Fraud_Rate (%)", ascending=False),
    x="Fraud_Rate (%)",
    y="merchant_city",
    palette=palette
)

ax.set_title("Top 10 Transaction Locations by Fraud Rate", fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel("Fraud Rate (%)", fontsize=12)
ax.set_ylabel("Transaction City", fontsize=12)

for i, v in enumerate(top10.sort_values("Fraud_Rate (%)", ascending=False)["Fraud_Rate (%)"]):
    ax.text(v + 0.5, i, f'{v:.2f}%', va='center', fontsize=10)

sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.show()

#### Fraud Rate by Transaction Type

> This is used to examine whether fraud rates differ between online and in-store transactions.

In [ ]:
online_summary = (
    train_df.assign(
        Transaction_Type=train_df["merchant_city"].eq("ONLINE")
                              .map({True: "ONLINE", False: "IN-STORE"})
    )
    .groupby("Transaction_Type")["label"]
    .agg(
        Transactions="count",
        Frauds="sum",
        Fraud_Rate="mean"
    )
    .reset_index()
)

online_summary["Fraud_Rate (%)"] = online_summary["Fraud_Rate"] * 100

print(online_summary)

In [ ]:

sns.set_theme(style="whitegrid", font_scale=1.1)

fig, ax = plt.subplots(figsize=(6,4.5))

palette = {"ONLINE": "#EF553B", "IN-STORE": "#636EFA"}
bars = ax.bar(
    online_summary["Transaction_Type"],
    online_summary["Fraud_Rate (%)"],
    color=[palette[t] for t in online_summary["Transaction_Type"]],
    width=0.5,
    edgecolor="white",
    linewidth=1.5
)

ax.set_title("Fraud Rate: ONLINE vs IN-STORE", fontsize=15, fontweight='bold', pad=15)
ax.set_ylabel("Fraud Rate (%)", fontsize=12)
ax.set_xlabel("")
ax.set_ylim(0, online_summary["Fraud_Rate (%)"].max() * 1.25)

for bar, rate in zip(bars, online_summary["Fraud_Rate (%)"]):
    ax.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + online_summary["Fraud_Rate (%)"].max()*0.03,
        f"{rate:.2f}%",
        ha="center", va="bottom", fontsize=11, fontweight='bold'
    )

sns.despine(left=True)
ax.grid(axis='y', linestyle='--', alpha=0.4)
ax.grid(axis='x', visible=False)

plt.tight_layout()
plt.show()

#### Transaction Time Intervals

> This indicates that fraudulent and legitimate transactions exhibit slightly different temporal patterns, with fraudulent transactions being relatively more concentrated within the 1–6 hour interval after a previous transaction.

In [ ]:
labels = [
    "<1 Hour",
    "1-6 Hours",
    "6-24 Hours",
    "1-7 Days",
    ">7 Days"
]
train_df = train_df.sort_values(["card_id", "date"])

train_df["time_since_prev"] = (
    train_df.groupby("card_id")["date"]
      .diff()
      .dt.total_seconds() / 60
)

bins = [-1, 60, 360, 1440, 10080, train_df["time_since_prev"].max()]

train_df["time_interval"] = pd.cut(
    train_df["time_since_prev"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

distribution = (
    train_df.groupby(["label", "time_interval"], observed=False)
            .size()
            .unstack(fill_value=0)
)

distribution_pct = distribution.div(distribution.sum(axis=1), axis=0) * 100
distribution_pct.index = ["Non-Fraud", "Fraud"]

print(distribution_pct.round(2))

ax = distribution_pct.T.plot(kind="bar", figsize=(8,5), width=0.8)
plt.title("Distribution of Time Since Previous Transaction")
plt.xlabel("Time Since Previous Transaction")
plt.ylabel("Percentage of Transactions (%)")
plt.legend(title="Class")

for container in ax.containers:
    ax.bar_label(container, fmt="%.1f%%", fontsize=9)

plt.tight_layout()
plt.show()

train_df = train_df.drop(columns='time_interval')

### Feature Engineering



#### Online Transaction

> Based on the previous EDA results, a new feature,`is_online_transaction` was created to identify whether a transaction was an online or in-store transaction.

In [ ]:
for df in [train_df, valid_df, test_df]:
    df["is_online_transaction"] = (
        df["merchant_city"] == "ONLINE"
    ).astype(int)

#### Time Since Previous Transaction & Within 6 Hours

> Based on the previous EDA results, the `time_since_prev` feature was created to calculate the time elapsed since the previous transaction, and the `within_6h` feature was derived to indicate whether the transaction occurred within six hours of the previous one.

In [ ]:
full_df = pd.concat([train_df, valid_df, test_df])
full_df = full_df.sort_values(["card_id", "date"])

full_df["time_since_prev"] = (
    full_df.groupby("card_id")["date"]
            .diff()
            .dt.total_seconds() / 60
)
full_df["time_since_prev"] = full_df["time_since_prev"].fillna(-1)
full_df["within_6h"] = (full_df["time_since_prev"] <= 360).astype(int)

train_df = full_df[full_df["date"].dt.year.between(2010, 2017)].copy()
valid_df = full_df[full_df["date"].dt.year == 2018].copy()
test_df  = full_df[full_df["date"].dt.year == 2019].copy()

In [ ]:
train_df.to_parquet(output_dir / "train.parquet", index=False)
valid_df.to_parquet(output_dir / "validation.parquet", index=False)
test_df.to_parquet(output_dir / "test.parquet", index=False)


### Load Data

In [ ]:
output_dir = Path("/content/drive/MyDrive/archive/data_split")

train_df = pd.read_parquet(output_dir / "train.parquet")
valid_df = pd.read_parquet(output_dir / "validation.parquet")
test_df = pd.read_parquet(output_dir / "test.parquet")

train_df_original = train_df.copy()
valid_df_original = valid_df.copy()
test_df_original = test_df.copy()

In [ ]:
drop_cols = [
    "id",
    "date",
    "client_id",
    "card_id",
    "card_brand",
    "card_type",
    "mcc_description",
    "gender",
    "retirement_age",
    "yearly_income",
    "merchant_id",
    "merchant_state",
    # "time_interval"
]

train_df = train_df.drop(columns=drop_cols)
valid_df = valid_df.drop(columns=drop_cols)
test_df = test_df.drop(columns=drop_cols)

### Encoding

In [ ]:
features = [col for col in train_df.columns if col != "label"]

print("Number of Features:", len(features))
print(features)


categorical_cols = train_df[features].select_dtypes(
    include=["object", "category"]
).columns

print("\nCategorical Features:")
print(categorical_cols.tolist())


encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

encoder.fit(train_df[categorical_cols])

train_df[categorical_cols] = encoder.transform(train_df[categorical_cols])
valid_df[categorical_cols] = encoder.transform(valid_df[categorical_cols])
test_df[categorical_cols] = encoder.transform(test_df[categorical_cols])

cat_mappings = {
    col: dict(enumerate(encoder.categories_[i]))
    for i, col in enumerate(categorical_cols)
}
train_df["label"] = train_df["label"].map({
    "Yes": 1,
    "No": 0
})
valid_df["label"] = valid_df["label"].map({
    "Yes": 1,
    "No": 0
})
test_df["label"] = test_df["label"].map({
    "Yes": 1,
    "No": 0
})
train_df["label"] = train_df["label"].astype(int)
valid_df["label"] = valid_df["label"].astype(int)
test_df["label"] = test_df["label"].astype(int)

X_train = train_df[features]
y_train = train_df["label"]

X_valid = valid_df[features]
y_valid = valid_df["label"]

X_test = test_df[features]
y_test = test_df["label"]


numeric_cols = [c for c in features if c not in categorical_cols.tolist()]

print("\nNumeric (scalable) Features:")
print(numeric_cols)
print("\nOrdinal-Encoded Categorical Features (not scaled):")
print(categorical_cols.tolist())

### Handle Imbalance Data

> Rather than oversampling or undersampling , the approach used class weighting because it preserves the original data distribution while encouraging the model to pay more attention to the minority class.

### BaseLine Comparison

> XGBoost was selected as the final model because it achieved the best overall performance across the evaluation metrics.

In [ ]:
baseline_results = []

scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_valid_scaled = X_valid.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_valid_scaled[numeric_cols] = scaler.transform(X_valid[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

#### Logistic Regression

In [ ]:
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()

class_weights = {
    0: 1,
    1: neg / pos
}

log_reg = LogisticRegression(
    class_weight=class_weights,
    max_iter=1000,
    solver="lbfgs",
    tol=1e-3,
    random_state=42,
    n_jobs=-1
)

log_reg.fit(X_train_scaled, y_train)

lr_valid_proba = log_reg.predict_proba(X_valid_scaled)[:, 1]
lr_valid_pred = log_reg.predict(X_valid_scaled)

print("\n Logistic Regression ")
print(classification_report(y_valid, lr_valid_pred))

baseline_results.append({
    "Model": "Logistic Regression",
    "F1": f1_score(y_valid, lr_valid_pred),
    "ROC-AUC": roc_auc_score(y_valid, lr_valid_proba),
    "PR-AUC": average_precision_score(y_valid, lr_valid_proba),
})

#### Random Forest

In [ ]:
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()

class_weights = {
    0: 1,
    1: neg / pos
}

rf = RandomForestClassifier(
    class_weight=class_weights,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

rf_valid_proba = rf.predict_proba(X_valid)[:, 1]
rf_valid_pred = rf.predict(X_valid)

print("\n Random Forest ")
print(classification_report(y_valid, rf_valid_pred))

baseline_results.append({
    "Model": "Random Forest",
    "F1": f1_score(y_valid, rf_valid_pred),
    "ROC-AUC": roc_auc_score(y_valid, rf_valid_proba),
    "PR-AUC": average_precision_score(y_valid, rf_valid_proba),
})

#### XGBoost

In [ ]:
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos if pos > 0 else 1

xgb_basic = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    scale_pos_weight=scale_pos_weight
)
xgb_basic.fit(X_train, y_train)

xgb_valid_proba = xgb_basic.predict_proba(X_valid)[:, 1]
xgb_valid_pred = (xgb_valid_proba >= 0.5).astype(int)

print("\n  XGBoost")
print(classification_report(y_valid, xgb_valid_pred))
baseline_results.append({
    "Model": "XGBoost",
    "F1": f1_score(y_valid, xgb_valid_pred),
    "ROC-AUC": roc_auc_score(y_valid, xgb_valid_proba),
    "PR-AUC": average_precision_score(y_valid, xgb_valid_proba),
})

In [ ]:
print("\n Baseline Comparison ")
baseline_df = pd.DataFrame(baseline_results).sort_values("PR-AUC", ascending=False)
print(baseline_df.to_string(index=False))


### HyperParameter Tuning

In [ ]:
from scipy.stats import randint, uniform
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos if pos > 0 else 1
xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    scale_pos_weight=scale_pos_weight,
    n_jobs=1,
    tree_method="hist"
)

param_dist = {
    "n_estimators": [100, 200],
    "max_depth": [4, 6, 8],
    "learning_rate": [0.05, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

random_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist,
    n_iter=20,
    scoring="f1",
    cv=cv,
    n_jobs=4,
    random_state=42,
    verbose=2
)

random_search.fit(X_train, y_train)

print("Best Parameters:")
print(random_search.best_params_)
print("Best CV F1:")
print(random_search.best_score_)

best_model = random_search.best_estimator_

In [ ]:
importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": best_model.feature_importances_
}).sort_values("Importance", ascending=False)

print(importance)

### Prediction

In [ ]:
valid_proba = best_model.predict_proba(X_valid)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_valid, valid_proba)
f1_scores = 2 * precision * recall / (precision + recall + 1e-12)
best_idx = np.argmax(f1_scores[:-1])
best_threshold = thresholds[best_idx]
print(f"Best threshold (by F1 on validation): {best_threshold:.4f}")

### Evaluation


Precision, Recall , F1 , ROC AOC

In [ ]:

y_valid_pred = (valid_proba >= best_threshold).astype(int)
print(classification_report(y_valid, y_valid_pred))
print(f1_score(y_valid, y_valid_pred))

test_proba = best_model.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= best_threshold).astype(int)

print("\nTest Results")
print(classification_report(y_test, test_pred))

In [ ]:

roc_auc = roc_auc_score(y_test, test_proba)
print(f"\nROC-AUC Score: {roc_auc:.4f}")

fpr, tpr, roc_thresholds = roc_curve(y_test, test_proba)

plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, label=f"ROC-AUC = {roc_auc:.4f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve (Test Set)")
plt.legend(loc="lower right")
plt.grid(True)
plt.show()


ConfusionMatrixDisplay.from_predictions(y_test, test_pred)
plt.title("Confusion Matrix (Test Set)")
plt.show()

In [ ]:

pr_auc = average_precision_score(y_test, test_proba)

print(f"PR-AUC Score: {pr_auc:.4f}")

In [ ]:
joblib.dump(best_model, "/content/drive/MyDrive/archive/xgboost_fraud_model.pkl")
joblib.dump(encoder, "/content/drive/MyDrive/archive/ordinal_encoder.pkl")
joblib.dump(features, "/content/drive/MyDrive/archive/feature_columns.pkl")
joblib.dump(best_threshold, "/content/drive/MyDrive/archive/best_threshold.pkl")


## SHAP

> SHAP value quantifies the contribution of a feature to the model's prediction.

In [ ]:
explainer = shap.TreeExplainer(best_model)

## LLM

> The LLM receives the transaction details, model prediction, and top SHAP feature contributions as input and produces a concise, human-readable explanation of the XGBoost model's prediction.

In [ ]:
best_model = joblib.load("/content/drive/MyDrive/archive/xgboost_fraud_model.pkl")
encoder = joblib.load("/content/drive/MyDrive/archive/ordinal_encoder.pkl")
features = joblib.load("/content/drive/MyDrive/archive/feature_columns.pkl")
best_threshold = joblib.load("/content/drive/MyDrive/archive/best_threshold.pkl")
explainer = shap.TreeExplainer(best_model)

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
MODEL_PATH = "/content/drive/MyDrive/HuggingFaceModels/Qwen2.5-1.5B-Instruct"

MODEL_CONFIG_FILE = os.path.join(MODEL_PATH, "config.json")

if not os.path.exists(MODEL_CONFIG_FILE):

    print("Downloading model...")

    os.makedirs(MODEL_PATH, exist_ok=True)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    model_llm = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map="auto",
        torch_dtype="auto"
    )

    tokenizer.save_pretrained(MODEL_PATH)
    model_llm.save_pretrained(MODEL_PATH)

    print("Model saved to Google Drive!")

else:
    print("Model already exists. Loading from Google Drive...")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

    model_llm = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        device_map="auto",
        torch_dtype="auto"
    )


generator = pipeline(
    "text-generation",
    model=model_llm,
    tokenizer=tokenizer
)


def build_mcc_description_map(df):
    mcc_col = next((c for c in df.columns if c.lower() == "mcc"), None)
    desc_col = next(
        (c for c in df.columns if "mcc" in c.lower() and "desc" in c.lower()),
        None,
    )

    if mcc_col is None or desc_col is None:
        raise ValueError(
            f"Could not find MCC code/description columns. "
            f"Found columns: {list(df.columns)}. "
            f"Adjust the matching logic above if your columns are named differently."
        )

    pairs = df[[mcc_col, desc_col]].dropna().drop_duplicates()
    mapping = (
        pairs.groupby(mcc_col)[desc_col]
        .agg(lambda s: s.value_counts().idxmax())
        .to_dict()
    )

    return {int(k): v for k, v in mapping.items()}
MCC_DESCRIPTIONS = build_mcc_description_map(train_df_original)


def decode_value(feature, value, cat_mappings):
    if feature == "mcc":
        try:
            code = int(value)
            description = MCC_DESCRIPTIONS.get(code, "Unknown category")
            return f"{code} - {description}"
        except (ValueError, TypeError):
            return value

    if feature == "zip":
        try:
            if float(value) == 0:
                return "Online transaction (no physical merchant ZIP)"
        except (ValueError, TypeError):
            pass
        return value

    if feature in cat_mappings:
        try:
            return cat_mappings[feature].get(int(value), value)
        except (ValueError, TypeError):
            return value

    return value


def explain_transaction(index):

    transaction = X_test.iloc[[index]]

    original = test_df_original.iloc[index]

    probability = best_model.predict_proba(transaction)[0][1]
    prediction = int(probability >= best_threshold)

    shap_result = explainer(transaction)

    sv = shap_result.values[0]
    if sv.ndim > 1:
        sv = sv[:, 1]
    explanation_df = pd.DataFrame({
        "Feature": transaction.columns,
        "Value": transaction.iloc[0].values,
        "SHAP": sv
    })
    explanation_df["ABS_SHAP"] = explanation_df["SHAP"].abs()

    explanation_df = (
        explanation_df
        .sort_values("ABS_SHAP", ascending=False)
        .head(5)
    )

    explanation_df["Value"] = explanation_df.apply(
        lambda r: decode_value(r["Feature"], r["Value"], cat_mappings), axis=1
    )

    print("\nTop Contributing Features")
    print(explanation_df)

    transaction_details = ""

    for col in original.index:
        if col != "label":
            decoded_val = decode_value(col, original[col], cat_mappings)
            transaction_details += f"- {col}: {decoded_val}\n"

    feature_text = ""

    for _, row in explanation_df.iterrows():

        direction = (
            "increased" if row["SHAP"] > 0
            else "decreased"
        )

        decoded_val = decode_value(row["Feature"], original[row["Feature"]], cat_mappings)

        feature_text += (
            f"Feature: {row['Feature']}\n"
            f"Current Value: {decoded_val}\n"
            f"Contribution: {direction}\n"
            f"SHAP Score: {row['SHAP']:.4f}\n\n"
        )

    system_prompt = (
        "You are a fraud investigation assistant supporting a bank's fraud monitoring team."
    )

    user_prompt = f"""
A machine learning model has already produced a prediction.
Your task is ONLY to explain the model's prediction using the provided SHAP values.
Do NOT evaluate whether the prediction is correct.

Rules:

- Use ONLY the transaction details and SHAP feature contributions provided.
- Do NOT invent facts, assumptions, or domain-specific explanations.
- SHAP values explain the model's decision, NOT the actual cause of fraud.
- A positive SHAP value means the feature pushed the prediction toward fraud.
- A negative SHAP value means the feature pushed the prediction toward legitimate.
- Do NOT state or imply that a feature is inherently suspicious.
- Do NOT describe values as "high", "low", "large", "small", "unusual", or "abnormal" unless those characteristics are explicitly provided.
- If the actual feature value is available, mention it. Otherwise, refer only to the feature name.
- If there is insufficient information to explain why a feature mattered, simply state that it contributed according to the model.
- Avoid causal language such as:
  * "because"
  * "indicates fraud"
  * "means the customer is risky"
  * "customers with lower credit limits are riskier"
  Instead use phrases such as:
  * "the model associated..."
  * "the model assigned greater importance..."
  * "according to the model..."
  * "this feature increased (or decreased) the predicted fraud probability."

Prediction:
{"Fraud" if prediction == 1 else "Legitimate"}

Fraud Probability:
{probability:.2%}

Transaction Details:
{transaction_details}

Top SHAP Feature Contributions:
{feature_text}

Generate the report in the following format.

### Summary
- State the model prediction.
- State the fraud probability.

### Key Factors
- List each important feature.
- State whether it increased or decreased the prediction.
- Include the SHAP contribution.
- Mention the feature value if available.

### Explanation
Explain that these features influenced the model's prediction according to SHAP.
Emphasize that the explanation reflects the model's learned patterns and should not be interpreted as proof that any feature causes fraud.

Limit the response to 150 words.
"""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    response = generator(
        messages,
        max_new_tokens=300,
        do_sample=False,
        return_full_text=False
    )

    print("LLM Explanation")

    print(response[0]["generated_text"].strip())

In [ ]:
fraud_index = test_df[test_df["label"] == 1].index[0]

position = test_df_original.index.get_loc(fraud_index)

explain_transaction(position)

In [ ]:
shap_values = explainer(X_test)
shap.summary_plot(shap_values.values, X_test)

In [ ]:
# import shutil

# shutil.rmtree(MODEL_PATH)